# Caso 5 — Temperatura estacionaria en una placa

Ángel Camilo Hincapié Quintero (2241945) · Samuel Alexis Gonzalez Martinez (2240680)

Bordes: izquierda 75 °C, derecha 50 °C, abajo 0 °C y arriba 100 °C. Se usa 1 m × 1 m como tamaño ilustrativo.


## 1. Datos del problema


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve
from pathlib import Path
from time import perf_counter
import csv, json

plt.rcParams.update({"figure.dpi": 115, "savefig.dpi": 190, "font.size": 10})
OUTPUT_DIR = Path("figuras_caso5")
OUTPUT_DIR.mkdir(exist_ok=True)

BORDES = {"izquierdo":75., "derecho":50., "inferior":0., "superior":100.}
Lx, Ly = 1.0, 1.0  # tamaño ilustrativo
print("Temperaturas de borde conocidas (°C):", BORDES)


## 2. Malla y sistema de ecuaciones


In [ ]:
def indice_interior(i, j, n):
    """Índice de los nodos interiores."""
    return (j - 1)*n + (i - 1)


def indice_global(i, j, n):
    """Índice global desde cero."""
    return i + j*(n + 2)


def indice_inverso(k, n, ny=None):
    """Coordenadas desde el índice global."""
    ny = n if ny is None else ny
    if k < 0 or k >= (n+2)*(ny+2):
        raise ValueError("k fuera de la malla")
    return k % (n+2), k // (n+2)


def ensamblar_sistema(n, bordes=BORDES, Lx=1.0, Ly=1.0, ny=None):
    """Arma el sistema A T = b con cinco puntos."""
    ny = n if ny is None else ny
    if not isinstance(n, (int, np.integer)) or n < 1 or not isinstance(ny, (int,np.integer)) or ny < 1:
        raise ValueError("n (nx) y ny deben ser enteros positivos")
    dx, dy = Lx/(n+1), Ly/(ny+1)
    r = (dx/dy)**2
    A = lil_matrix((n*ny, n*ny), dtype=float)
    b = np.zeros(n*ny, dtype=float)
    for j in range(1,ny+1):
        for i in range(1,n+1):
            p = indice_interior(i,j,n)
            A[p,p] = 2*(1+r)
            if i == 1:
                b[p] += bordes["izquierdo"]
            else:
                A[p,indice_interior(i-1,j,n)] = -1
            if i == n:
                b[p] += bordes["derecho"]
            else:
                A[p,indice_interior(i+1,j,n)] = -1
            if j == 1:
                b[p] += r*bordes["inferior"]
            else:
                A[p,indice_interior(i,j-1,n)] = -r
            if j == ny:
                b[p] += r*bordes["superior"]
            else:
                A[p,indice_interior(i,j+1,n)] = -r
    return A.tocsr(),b


def reconstruir(x, n, bordes=BORDES, ny=None):
    """Reconstruye la matriz de temperaturas."""
    ny = n if ny is None else ny
    U = np.zeros((ny+2,n+2), dtype=float)
    x = np.asarray(x)
    if x.size != n*ny:
        raise ValueError("El vector solución no coincide con la malla")
    for p, valor in enumerate(x):
        i = p % n + 1
        j = p // n + 1
        k = indice_global(i, j, n)
        i_rec, j_rec = indice_inverso(k, n, ny)
        assert (i_rec, j_rec) == (i, j)
        U[j_rec, i_rec] = valor
    U[1:-1,0] = bordes["izquierdo"]
    U[1:-1,-1] = bordes["derecho"]
    U[0,1:-1] = bordes["inferior"]
    U[-1,1:-1] = bordes["superior"]
    U[0,0] = (bordes["inferior"]+bordes["izquierdo"])/2
    U[0,-1] = (bordes["inferior"]+bordes["derecho"])/2
    U[-1,0] = (bordes["superior"]+bordes["izquierdo"])/2
    U[-1,-1] = (bordes["superior"]+bordes["derecho"])/2
    return U

A3,b3 = ensamblar_sistema(3)
print("Número de incógnitas:",A3.shape[0])
print("A (malla 5 x 5):\n",A3.toarray().astype(int))
print("b:",b3)
print("Numeración nodos interiores k:",[
    indice_global(i,j,3) for j in range(1,4) for i in range(1,4)])


## 3. Numeración de los nodos


In [ ]:
from matplotlib.patches import Patch

def clasificar_nodo(i, j, nx, ny):
    lados=[]
    if i == 0: lados.append("izquierdo")
    if i == nx+1: lados.append("derecho")
    if j == 0: lados.append("inferior")
    if j == ny+1: lados.append("superior")
    if len(lados) == 0:
        return "interior"
    if len(lados) == 2:
        return "esquina"
    return lados[0]

nx0=ny0=3
reg_nodos=[]
for j in range(ny0+2):
    for i in range(nx0+2):
        k=indice_global(i,j,nx0)
        ir,jr=indice_inverso(k,nx0)
        assert (ir,jr)==(i,j)
        reg_nodos.append((k,i,j,clasificar_nodo(i,j,nx0,ny0)))
assert len(reg_nodos)==25
assert sum(r[3]=="interior" for r in reg_nodos)==9
assert sum(r[3]!="interior" for r in reg_nodos)==16
print("k  i  j  tipo")
for k,i,j,tipo in reg_nodos:
    print(f"{k:2d} {i:2d} {j:2d}  {tipo}")
print("Inversión ejemplar:", {k:indice_inverso(k,nx0) for k in (6,7,8,11,12,13,16,17,18)})
with open(OUTPUT_DIR/"tabla_nodos_5x5.csv", "w", newline="",encoding="utf-8") as f:
    wr=csv.writer(f)
    wr.writerow(["k_global","i","j","tipo"])
    wr.writerows(reg_nodos)

fig,ax=plt.subplots(figsize=(8,7.3))
for v in range(5):
    ax.plot([0,4],[v,v],color="0.83",lw=1,zorder=0)
    ax.plot([v,v],[0,4],color="0.83",lw=1,zorder=0)
col={"interior":"#3468a5", "superior":"#c94e48", "inferior":"#4385b9",
     "izquierdo":"#c79c3a", "derecho":"#429879", "esquina":"#777777"}
for k,i,j,tipo in reg_nodos:
    ax.scatter(i,j,s=720,c=col[tipo],edgecolor="white",linewidth=2.2,zorder=3)
    ax.text(i,j+0.06,f"{k}",color="white",ha="center",va="center",fontsize=13,fontweight="bold",zorder=4)
    if tipo=="interior":
        ax.text(i,j-0.22,f"T{str(k)}",color="white",ha="center",va="center",fontsize=8.5,zorder=4)
ax.set(xlim=(-.65,4.65),ylim=(-.75,4.65),title="Numeración global k=i+5j y clasificación de nodos",
       xlabel="Índice horizontal i",ylabel="Índice vertical j")
ax.set_aspect("equal")
ax.set_xticks(range(5));ax.set_yticks(range(5))
ax.legend(handles=[Patch(color=col[x],label=x.capitalize()) for x in
          ("interior","izquierdo","derecho","inferior","superior","esquina")],
          loc="upper center",bbox_to_anchor=(.5,-.10),ncol=3,frameon=False,fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"fig07_numeracion_global.png",dpi=220,bbox_inches="tight")
plt.show()


## 4. Solución directa (malla 5 × 5)


In [ ]:
t0 = perf_counter()
x_directa = np.linalg.solve(A3.toarray(), b3)
tiempo_directo_3 = perf_counter()-t0
U3 = reconstruir(x_directa,3)
residuo_directo_3 = np.linalg.norm(A3@x_directa-b3, ord=np.inf)

print("Solución de 9 temperaturas, orden del enunciado (°C):")
for j in range(1,4):
    for i in range(1,4):
        p=indice_interior(i,j,3)
        print(f"T_{indice_global(i,j,3):2d} = {x_directa[p]:10.6f} °C")
print("Matriz U interior (fila inferior -> superior):\n",np.round(U3[1:-1,1:-1],6))
print(f"Residuo ||Ax-b||inf = {residuo_directo_3:.2e}")
assert residuo_directo_3<1e-10
assert U3[1:-1,1:-1].min()>=min(BORDES.values())-1e-10
assert U3[1:-1,1:-1].max()<=max(BORDES.values())+1e-10
error_armonico = np.max(np.abs(
    U3[1:-1,1:-1] - (U3[1:-1,:-2]+U3[1:-1,2:]+U3[:-2,1:-1]+U3[2:,1:-1])/4))
print(f"Error de promedio de vecinos máximo = {error_armonico:.2e} °C")

fig, ax = plt.subplots(figsize=(7.0,5.7))
coords_x=np.linspace(0,Lx,5)
coords_y=np.linspace(0,Ly,5)
mesh=ax.pcolormesh(coords_x,coords_y,U3,shading="gouraud",cmap="inferno",vmin=0,vmax=120)
for j in range(5):
    for i in range(5):
        if (i in (0,4)) and (j in (0,4)):
            continue  # esquinas sin temperatura única
        x0, y0=coords_x[i],coords_y[j]
        ax.plot([x0],[y0],"o",ms=3,color="white",alpha=.9)
        ax.text(x0,y0,f"{U3[j,i]:.1f}",ha="center",va="center",fontsize=8.5,
                color="white" if U3[j,i]<60 else "black",
                bbox={"facecolor":"black" if U3[j,i]<60 else "white","alpha":0.72,
                      "edgecolor":"none","pad":1.1})
ax.set(xlim=(-.07,Lx+.07),ylim=(-.07,Ly+.07),xlabel="x (m)",ylabel="y (m)",
       title="Malla 5×5: temperaturas nodales (°C)")
ax.set_aspect("equal")
fig.colorbar(mesh,ax=ax,label="Temperatura (°C)",shrink=.85)
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"fig01_malla_5x5.png",bbox_inches="tight")
plt.show()


## 5. Reconstrucción de la malla


In [ ]:
mapeo=[]
for p,t in enumerate(x_directa):
    ii=p%3+1
    jj=p//3+1
    kk=indice_global(ii,jj,3)
    ir,jr=indice_inverso(kk,3)
    assert (ir,jr)==(ii,jj)
    mapeo.append((kk,ir,jr,t))
assert np.allclose(U3[1:-1,1:-1], x_directa.reshape(3,3))
print("k global → (i,j) → temperatura (°C)")
for kk,ir,jr,t in mapeo:
    print(f"T_{kk:2d} → ({ir},{jr}) → {t:9.5f}")
fig,axs=plt.subplots(1,2,figsize=(11,4.9),gridspec_kw={"width_ratios":[.8,1.1]},layout="constrained")
axs[0].set_xlim(0,1);axs[0].set_ylim(-.5,9);axs[0].axis("off")
axs[0].set_title("Vector ordenado por k global",fontsize=12)
for p,(kk,ir,jr,t) in enumerate(mapeo):
    y=8-p
    axs[0].text(.08,y,f"T$_{{{kk}}}$",fontsize=11,va="center",color="#3468a5",weight="bold")
    axs[0].text(.32,y,f"k={kk:2d} → ({ir},{jr})",fontsize=10,va="center")
    axs[0].text(.95,y,f"{t:.2f} °C",fontsize=10,va="center",ha="right")
axs[1].imshow(U3[1:-1,1:-1],origin="lower",cmap="YlOrRd",vmin=0,vmax=100,extent=(.5,3.5,.5,3.5))
axs[1].set(title="Matriz reconstruida U[j,i]",xlabel="Índice i",ylabel="Índice j",xticks=[1,2,3],yticks=[1,2,3]);axs[1].set_aspect("equal")
for kk,ir,jr,t in mapeo:
    axs[1].text(ir,jr,f"T$_{{{kk}}}$\n{t:.2f}",ha="center",va="center",fontsize=10,color="black")
fig.savefig(OUTPUT_DIR/"fig08_reconstruccion_inversa.png",dpi=220,bbox_inches="tight")
plt.show()


## 6. Jacobi, Gauss–Seidel y SOR


In [ ]:
def iterar(A,b,metodo="jacobi",omega=1.,tol=1e-10,maxiter=50000):
    """Métodos iterativos."""
    A=A.tocsr()
    D=A.diagonal()
    x=np.zeros_like(b,dtype=float)
    scale=max(1.0,np.linalg.norm(b,np.inf))
    residuals=[]
    t0=perf_counter()
    for it in range(1,maxiter+1):
        if metodo=="jacobi":
            xnew=(b-(A@x-D*x))/D
            x=xnew
        elif metodo in ("gauss-seidel","sor"):
            for p in range(len(b)):
                start,end=A.indptr[p],A.indptr[p+1]
                ss=0.0
                for q,v in zip(A.indices[start:end],A.data[start:end]):
                    if q != p:
                        ss+=v*x[q]
                gs=(b[p]-ss)/D[p]
                x[p]=(1-omega)*x[p]+omega*gs if metodo=="sor" else gs
        else:
            raise ValueError("Método desconocido")
        rr=np.linalg.norm(A@x-b,np.inf)/scale
        residuals.append(rr)
        if rr < tol:
            return {"x":x,"iteraciones":it,"residuo":rr,
                    "tiempo_s":perf_counter()-t0,"historial":np.asarray(residuals)}
    raise RuntimeError(f"{metodo}: no convergió en {maxiter} iteraciones")

omega_sor=2/(1+np.sin(np.pi/4))
metodos={
    "Jacobi":iterar(A3,b3,"jacobi"),
    "Gauss-Seidel":iterar(A3,b3,"gauss-seidel"),
    "SOR":iterar(A3,b3,"sor",omega=omega_sor)
}
print(f"SOR: omega = {omega_sor:.7f}")
print(f"{'Método':<17}{'Iteraciones':>13}{'Residuo relativo':>20}{'Error max vs directa (°C)':>30}")
for nombre,sol in metodos.items():
    err=np.max(np.abs(sol["x"]-x_directa))
    print(f"{nombre:<17}{sol['iteraciones']:>13d}{sol['residuo']:>20.3e}{err:>30.3e}")

fig,ax=plt.subplots(figsize=(7.0,4.5))
for nombre,sol in metodos.items():
    ax.semilogy(np.arange(1,len(sol["historial"])+1),sol["historial"],label=nombre)
ax.axhline(1e-10,ls="--",c="gray",lw=1,label="Tolerancia")
ax.set(xlabel="Número de iteración",ylabel="Residuo relativo (norma infinito)",
       title="Convergencia de los métodos iterativos — 9 incógnitas")
ax.legend();ax.grid(alpha=.3)
fig.tight_layout();fig.savefig(OUTPUT_DIR/"fig02_convergencia_iterativos.png",bbox_inches="tight");plt.show()


## 7. Distribución de temperatura


In [ ]:
n_fino=63
Af,bf=ensamblar_sistema(n_fino)
t0=perf_counter()
xf=spsolve(Af,bf)
tiempo_fino=perf_counter()-t0
Uf=reconstruir(xf,n_fino)
xfis=np.linspace(0,Lx,n_fino+2)
yfis=np.linspace(0,Ly,n_fino+2)
X,Y=np.meshgrid(xfis,yfis)
grad_y,grad_x=np.gradient(Uf,yfis,xfis,edge_order=2)
fig,axs=plt.subplots(1,2,figsize=(11.6,4.8),layout="constrained")
im=axs[0].pcolormesh(X,Y,Uf,shading="auto",cmap="inferno",vmin=0,vmax=120)
axs[0].contour(X,Y,Uf,levels=np.arange(10,101,10),colors="white",linewidths=.55,alpha=.8)
axs[0].set(title="Mapa de calor e isotermas — 63×63 interiores",xlabel="x (m)",ylabel="y (m)")
fig.colorbar(im,ax=axs[0],label="Temperatura (°C)")
step=5
axs[1].contourf(X,Y,Uf,levels=24,cmap="YlOrRd")
axs[1].quiver(X[2:-2:step,2:-2:step],Y[2:-2:step,2:-2:step],
              -grad_x[2:-2:step,2:-2:step],-grad_y[2:-2:step,2:-2:step],
              color="black",scale=2600,width=.003)
axs[1].set(title="Dirección del flujo relativo: $-\nabla T$",xlabel="x (m)",ylabel="y (m)")
fig.savefig(OUTPUT_DIR/"fig03_campo_y_flujo.png",bbox_inches="tight");plt.show()
print("Residuo fino inf:",np.linalg.norm(Af@xf-bf,np.inf),"; tiempo s:",tiempo_fino)
print("Centro fino (°C):",Uf[(n_fino+1)//2,(n_fino+1)//2])


## 8. Cambio de temperatura en el borde superior


In [ ]:
BORDES_CAMBIO = {**BORDES,"superior":120.}
Aalt,balt=ensamblar_sistema(n_fino,BORDES_CAMBIO)
xalt=spsolve(Aalt,balt)
Ualt=reconstruir(xalt,n_fino,BORDES_CAMBIO)
Delta=Ualt-Uf
c=(n_fino+1)//2
print(f"Temperatura central original: {Uf[c,c]:.6f} °C")
print(f"Temperatura central modificada: {Ualt[c,c]:.6f} °C")
print(f"Aumento central: {Delta[c,c]:.6f} °C")
fig,axs=plt.subplots(1,3,figsize=(14,4),layout="constrained")
for ax,U,titulo in zip(axs[:2],[Uf,Ualt],["Caso base: superior 100 °C","Cambio: superior 120 °C"]):
    m=ax.imshow(U,origin="lower",extent=(0,Lx,0,Ly),cmap="inferno",vmin=0,vmax=120)
    ax.set(title=titulo,xlabel="x (m)",ylabel="y (m)")
md=axs[2].imshow(Delta,origin="lower",extent=(0,Lx,0,Ly),cmap="viridis",vmin=0,vmax=20)
axs[2].set(title="Aumento local de temperatura (°C)",xlabel="x (m)",ylabel="y (m)")
fig.colorbar(m,ax=axs[:2],shrink=.8,label="Temperatura (°C)")
fig.colorbar(md,ax=axs[2],shrink=.8,label="ΔT (°C)")
fig.savefig(OUTPUT_DIR/"fig04_modificacion_frontera.png",bbox_inches="tight");plt.show()


## 9. Comparación de gradientes


In [ ]:
dy=Ly/(n_fino+1)
dx=Lx/(n_fino+1)
g0y,g0x=np.gradient(Uf,dy,dx,edge_order=2)
g1y,g1x=np.gradient(Ualt,dy,dx,edge_order=2)
mag0=np.hypot(g0x,g0y)
mag1=np.hypot(g1x,g1y)
dmag=mag1-mag0
dgx=g1x-g0x
dgy=g1y-g0y
# Evitar las esquinas al comparar gradientes.
xx=np.linspace(0,Lx,n_fino+2)
yy=np.linspace(0,Ly,n_fino+2)
M=(xx>=.2*Lx)&(xx<=.8*Lx)
N=(yy>=.2*Ly)&(yy<=.8*Ly)
core=np.ix_(N,M)
centro=(c,c)
medidas_grad={
    "base_gx_centro_C_por_m":float(g0x[centro]),
    "base_gy_centro_C_por_m":float(g0y[centro]),
    "cambio_gx_centro_C_por_m":float(g1x[centro]),
    "cambio_gy_centro_C_por_m":float(g1y[centro]),
    "base_norma_centro_C_por_m":float(mag0[centro]),
    "cambio_norma_centro_C_por_m":float(mag1[centro]),
    "delta_norma_centro_C_por_m":float(dmag[centro]),
    "base_media_norma_centro_region_C_por_m":float(np.mean(mag0[core])),
    "cambio_media_norma_centro_region_C_por_m":float(np.mean(mag1[core])),
    "delta_media_norma_centro_region_C_por_m":float(np.mean(dmag[core])),
    "media_norma_delta_gradiente_region_C_por_m":float(np.mean(np.hypot(dgx[core],dgy[core]))),
}
for clave,valor in medidas_grad.items():print(f"{clave}: {valor:.6f}")
assert np.allclose(g1x[centro],g0x[centro],atol=1e-8)
assert g1y[centro]>g0y[centro]

from matplotlib.colors import TwoSlopeNorm
Xc,Yc=np.meshgrid(xx[M],yy[N])
G0,G1,D=mag0[core],mag1[core],dmag[core]
fig,axs=plt.subplots(2,2,figsize=(11.5,8.5),layout="constrained")
vmax=float(max(np.max(G0),np.max(G1)))
for ax,arr,title in ((axs[0,0],G0,"Caso base: |∇T|"),(axs[0,1],G1,"Borde superior a 120 °C: |∇T|")):
    hm=ax.pcolormesh(Xc,Yc,arr,cmap="magma",vmin=0,vmax=vmax,shading="auto")
    ax.set(title=title,xlabel="x (m, ilustrativo)",ylabel="y (m, ilustrativo)",aspect="equal")
fig.colorbar(hm,ax=[axs[0,0],axs[0,1]],label="Magnitud del gradiente (°C/m)",shrink=.81)
R=float(max(abs(D.min()),abs(D.max())))
ht=axs[1,0].pcolormesh(Xc,Yc,D,cmap="RdBu_r",norm=TwoSlopeNorm(vmin=-R,vcenter=0,vmax=R),shading="auto")
axs[1,0].set(title="Cambio en magnitud: |∇T| nuevo − |∇T| base",xlabel="x (m, ilustrativo)",ylabel="y (m, ilustrativo)",aspect="equal")
fig.colorbar(ht,ax=axs[1,0],label="Cambio de |∇T| (°C/m)")
perfil=(xx>=.2*Lx)&(xx<=.8*Lx)
axs[1,1].plot(xx[perfil], g0y[c,perfil],label="Base",lw=2)
axs[1,1].plot(xx[perfil], g1y[c,perfil],label="Superior 120 °C",lw=2,ls="--")
axs[1,1].set(title="Componente vertical ∂T/∂y a media altura",xlabel="x (m, ilustrativo)",ylabel="∂T/∂y (°C/m)")
axs[1,1].legend();axs[1,1].grid(alpha=.3)
fig.savefig(OUTPUT_DIR/"fig09_comparacion_gradientes.png",dpi=220,bbox_inches="tight")
plt.show()

sel=(yy>=.1*Ly)&(yy<=.9*Ly)
fig,axs=plt.subplots(1,2,figsize=(11.4,4.25),layout="constrained")
axs[0].plot(yy[sel],g0y[sel,c],label="Base",lw=2)
axs[0].plot(yy[sel],g1y[sel,c],label="Superior 120 °C",lw=2,ls="--")
axs[0].set(title="Gradiente vertical en x=0.5 m",xlabel="y (m, ilustrativo)",ylabel="∂T/∂y (°C/m)")
axs[0].legend();axs[0].grid(alpha=.3)
axs[1].plot(yy[sel],dgy[sel,c],color="#8c4b90",lw=2,label="Δ(∂T/∂y)")
axs[1].axhline(0,color="0.4",lw=1)
axs[1].set(title="Cambio del gradiente vertical",xlabel="y (m, ilustrativo)",ylabel="Δ(∂T/∂y) (°C/m)")
axs[1].grid(alpha=.3)
fig.savefig(OUTPUT_DIR/"fig10_perfil_gradiente.png",dpi=220,bbox_inches="tight")
plt.show()

with open(OUTPUT_DIR/"tabla_comparacion_gradientes.csv", "w",newline="",encoding="utf-8") as f:
    writer=csv.writer(f)
    writer.writerow(["magnitud","valor"])
    writer.writerows(medidas_grad.items())


## 10. Refinamiento de malla


In [ ]:
tam=[3,7,15,31,63,127]
soluciones={}
tiempos={}
for n in tam:
    AA,bb=ensamblar_sistema(n)
    start=perf_counter()
    sol=spsolve(AA,bb)
    tiempos[n]=perf_counter()-start
    soluciones[n]=reconstruir(sol,n)
    rr=np.linalg.norm(AA@sol-bb,np.inf)
    print(f"n={n:3d} | incógnitas={n*n:6d} | solve={tiempos[n]:9.5f} s | residuo_inf={rr:.2e}")

ref=soluciones[127]
registros=[]
for n in tam[:-1]:
    stride=(127+1)//(n+1)
    ref_comun=ref[::stride,::stride]
    assert ref_comun.shape==soluciones[n].shape
    diff=soluciones[n][1:-1,1:-1]-ref_comun[1:-1,1:-1]
    registros.append({"n_interior":n,"incognitas":n*n,"h_m":Lx/(n+1),
                      "error_max_C":float(np.max(np.abs(diff))),
                      "error_RMS_C":float(np.sqrt(np.mean(diff**2))),
                      "tiempo_s":float(tiempos[n])})
print("\nComparación con referencia n=127 (nodos interiores comunes):")
for r in registros:
    print("n={n_interior:3d}, h={h_m:.6f} m, max={error_max_C:.5f} °C, RMS={error_RMS_C:.5f} °C, t={tiempo_s:.5f} s".format(**r))

fig,axs=plt.subplots(1,2,figsize=(11.4,4.2),layout="constrained")
axs[0].loglog([r["h_m"] for r in registros],[r["error_max_C"] for r in registros],"o-",label="Error máximo")
axs[0].loglog([r["h_m"] for r in registros],[r["error_RMS_C"] for r in registros],"s--",label="Error RMS")
axs[0].set(xlabel="Paso espacial h (m)",ylabel="Diferencia con referencia (°C)",
           title="Diferencias entre mallas anidadas")
axs[0].legend();axs[0].grid(alpha=.3,which="both")
axs[1].plot([n*n for n in tam],[tiempos[n] for n in tam],"o-")
axs[1].set(xlabel="Número de incógnitas ($n^2$)",ylabel="Tiempo de resolución (s)",
           title="Costo de la solución directa dispersa")
axs[1].grid(alpha=.3)
fig.savefig(OUTPUT_DIR/"fig05_refinamiento_y_costo.png",bbox_inches="tight");plt.show()

with open(OUTPUT_DIR/"tabla_refinamiento.csv","w",newline="",encoding="utf-8") as f:
    dw=csv.DictWriter(f,fieldnames=list(registros[0].keys()))
    dw.writeheader();dw.writerows(registros)


## 11. Malla rectangular


In [ ]:
nx_rect,ny_rect=9,5
Lx_rect,Ly_rect=2.0,1.0
Arect,brect=ensamblar_sistema(nx_rect,Lx=Lx_rect,Ly=Ly_rect,ny=ny_rect)
sol_rect=spsolve(Arect,brect)
Urect=reconstruir(sol_rect,nx_rect,ny=ny_rect)
assert Urect.shape==(ny_rect+2,nx_rect+2)
assert np.linalg.norm(Arect@sol_rect-brect,np.inf)<1e-9
assert 0 <= Urect[1:-1,1:-1].min() <= Urect[1:-1,1:-1].max() <= 100
print(f"Rectángulo {Lx_rect} × {Ly_rect} m: nodos interiores {nx_rect}×{ny_rect}, incógnitas {len(sol_rect)}")
print("Rango interior (°C):",Urect[1:-1,1:-1].min(),Urect[1:-1,1:-1].max())
fig,ax=plt.subplots(figsize=(8.6,3.7))
Xr,Yr=np.meshgrid(np.linspace(0,Lx_rect,nx_rect+2),np.linspace(0,Ly_rect,ny_rect+2))
mapr=ax.pcolormesh(Xr,Yr,Urect,shading="gouraud",cmap="inferno",vmin=0,vmax=100)
ax.contour(Xr,Yr,Urect,levels=np.arange(10,100,10),colors="white",linewidths=.6)
ax.set(title="Prueba adicional: malla rectangular 9×5 interiores",xlabel="x (m)",ylabel="y (m)")
fig.colorbar(mapr,ax=ax,label="Temperatura (°C)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"fig06_rectangular.png",bbox_inches="tight")
plt.show()


## 12. Guardar resultados


In [ ]:
assert np.allclose(A3.toarray(),A3.toarray().T)
assert np.all(np.linalg.eigvalsh(A3.toarray())>0)
assert np.max(np.abs(A3@x_directa-b3))<1e-10
assert np.allclose(Uf[c,c],56.25,atol=1e-9)
assert np.allclose(Delta[c,c],5.0,atol=1e-9)
assert np.all((Uf[1:-1,1:-1]>=0)&(Uf[1:-1,1:-1]<=100))
assert all(np.allclose(sol["x"],x_directa,atol=1e-7) for sol in metodos.values())
assert len(reg_nodos)==25 and len(mapeo)==9
assert np.allclose(U3[1:-1,1:-1],x_directa.reshape(3,3))

np.savetxt(OUTPUT_DIR/"temperaturas_5x5.csv",U3,delimiter=",",fmt="%.9f")
np.savetxt(OUTPUT_DIR/"temperaturas_65x65.csv",Uf,delimiter=",",fmt="%.9f")
resultados={
    "nodos_5x5_orden_global":{str(indice_global(i,j,3)):float(x_directa[indice_interior(i,j,3)])
                               for j in range(1,4) for i in range(1,4)},
    "residuo_directo_inf":float(residuo_directo_3),
    "error_armonico_5x5":float(error_armonico),
    "omega_SOR":float(omega_sor),
    "comparacion_metodos":{nombre:{"iteraciones":int(d["iteraciones"]),
                                   "residuo":float(d["residuo"]),
                                   "max_diferencia_con_directo_C":float(np.max(np.abs(d["x"]-x_directa)))}
                           for nombre,d in metodos.items()},
    "caso_base_centro_C":float(Uf[c,c]),
    "escenario_centro_C":float(Ualt[c,c]),
    "delta_centro_C":float(Delta[c,c]),
    "refinamiento":registros,
    "comparacion_gradientes": medidas_grad,
    "tiempo_referencia_127_s":float(tiempos[127])
}
(OUTPUT_DIR/"resumen_resultados.json").write_text(json.dumps(resultados,ensure_ascii=False,indent=2),encoding="utf-8")
print("Verificaciones superadas.")
print("Figuras y tablas guardadas en:",OUTPUT_DIR.resolve())
print("Archivos:",[p.name for p in sorted(OUTPUT_DIR.iterdir())])
